# Healthcare Agent Learning Lab 9/10

## Agent tools over a FHIR sandbox

**Suggested time:** 30 minutes  
**Demonstration type:** In-notebook synthetic FHIR mock with optional HAPI adapter

### Learning objectives

- Identify how FHIR resources become agent tools and observations.
- Separate patient state, tool interface, agent policy, and write-back controls.
- Recognize the additional work required for a governed EHR integration.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** move the toy tools of NB 2/NB 3 onto a real **FHIR** interface — the agent reads an INR and medications as FHIR resources and writes a flag back — while the *loop and the gate stay identical.* This is the bridge from teaching scaffold to the virtual patient environment.

It ships a **mock FHIR client** so it runs with no infrastructure. To use real data, generate patients with **Synthea**, load them into a **HAPI FHIR** server, and set `FHIR_BASE_URL` — the agent code doesn't change. (Runs in MOCK mode with no API key.)


> ### In plain terms
> This is the *same* agent from earlier, but instead of toy data it reads and writes a patient the way a real hospital system does — through a standard called **FHIR**. The whole point: the agent's logic doesn't change, only the data source does — so what you prototype on synthetic data runs against a real electronic health record later, unchanged.
>
> **What you'll see:** the same loop and the same human gate as before; only the tools now speak FHIR. Watch it read the INR, get **gated** on the write-back, then read back what it wrote to the record.


In [ ]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    "Scripted ReAct actions for the FHIR-backed anticoagulant loop."
    hist = " ".join(m["content"] for m in messages if m["role"] != "system")
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Read the latest INR via FHIR.","action":"get_latest_inr","action_input":{}})
    if "get_medications" not in hist:
        return json.dumps({"thought":"Check active medications.","action":"get_medications","action_input":{}})
    if "write_flag" not in hist:
        return json.dumps({"thought":"INR 4.2 is out of range — draft a dose-review flag for the clinician.",
                           "action":"write_flag","action_input":{"text":"INR 4.2 above range; request dose review + earlier follow-up."}})
    return json.dumps({"thought":"Flag written and awaiting review.","action":"final","action_input":{"summary":"Drafted dose-review flag; no order changed."}})


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### A synthetic FHIR patient + a FHIR client
The local client illustrates the FHIR REST surface using Observation, MedicationRequest, and Communication resources. External access is disabled by default. The optional adapter accepts only an explicitly selected localhost sandbox, requires a patient ID, and keeps writes disabled unless separately enabled.


In [ ]:
# A Synthea-style FHIR bundle (trimmed). LOINC 6301-6 = INR; RxNorm 855332 = warfarin 5 mg.
BUNDLE = {
 "Observation": [{"code":"6301-6","display":"INR","valueQuantity":{"value":4.2,"unit":"ratio"},
                  "referenceRange":"2.0-3.0","effectiveDateTime":"today"}],
 "MedicationRequest": [{"medication":"warfarin 5 mg oral tablet","rxnorm":"855332","status":"active"}],
 "Communication": [],   # write-backs land here
}
from urllib.parse import urlparse

FHIR_MODE = os.environ.get("FHIR_MODE", "local").strip().lower()
if FHIR_MODE not in {"local", "sandbox"}:
    raise RuntimeError("FHIR_MODE must be 'local' or 'sandbox'.")
BASE = None
FHIR_PATIENT_ID = None
ALLOW_FHIR_WRITE = os.environ.get("ALLOW_FHIR_WRITE", "false").lower() == "true"
if FHIR_MODE == "sandbox":
    BASE = os.environ.get("FHIR_BASE_URL")
    FHIR_PATIENT_ID = os.environ.get("FHIR_PATIENT_ID")
    if not BASE or not FHIR_PATIENT_ID:
        raise RuntimeError("FHIR_MODE=sandbox requires FHIR_BASE_URL and FHIR_PATIENT_ID.")
    if urlparse(BASE).hostname not in {"localhost", "127.0.0.1", "::1"}:
        raise RuntimeError("This teaching notebook only permits a localhost FHIR sandbox.")

class FHIR:
    def get_latest_inr(self, **_):
        if BASE:
            import requests
            r = requests.get(f"{BASE}/Observation", params={"patient":FHIR_PATIENT_ID,
                             "code":"http://loinc.org|6301-6", "_sort":"-date",
                             "_count":1}, timeout=10).json()
            o = r["entry"][0]["resource"]
            return {"inr":o["valueQuantity"]["value"], "reference_range":"2.0-3.0"}
        o = BUNDLE["Observation"][0]
        return {"inr":o["valueQuantity"]["value"], "reference_range":o["referenceRange"]}
    def get_medications(self, **_):
        if BASE:
            import requests
            r = requests.get(f"{BASE}/MedicationRequest", params={"patient":FHIR_PATIENT_ID,
                             "status":"active"}, timeout=10).json()
            return {"medications":[e["resource"]["medicationCodeableConcept"]["text"] for e in r.get("entry",[])]}
        return {"medications":[m["medication"] for m in BUNDLE["MedicationRequest"]]}
    def write_flag(self, **a):
        text = next((a[k] for k in ("text","reason","summary","note","content","flag") if a.get(k)),
                    next((v for v in a.values() if isinstance(v,str) and v.strip()), ""))
        if BASE:
            import requests
            if not ALLOW_FHIR_WRITE:
                return {"blocked":"external FHIR writes are disabled; set ALLOW_FHIR_WRITE=true only for a disposable localhost sandbox"}
            payload={"resourceType":"Communication","status":"preparation",
                     "subject":{"reference":f"Patient/{FHIR_PATIENT_ID}"},
                     "payload":[{"contentString":text}]}
            response = requests.post(f"{BASE}/Communication", json=payload, timeout=10)
            response.raise_for_status()
        BUNDLE["Communication"].append({"status":"preparation","payload":text})
        return {"written":text}

fhir = FHIR()
print("FHIR backend:", "REAL HAPI @ "+BASE if BASE else "MOCK (in-notebook synthetic bundle)")
print("Sanity read ->", fhir.get_latest_inr())


### Tools wrap FHIR calls; the write-back is gated
Identical loop to NB 2, identical gate to NB 3 — only the tool bodies changed from a dict to FHIR calls.


In [ ]:
TOOLS = {"get_latest_inr":fhir.get_latest_inr, "get_medications":fhir.get_medications, "write_flag":fhir.write_flag}
GATED = {"write_flag"}                      # writing to the record is consequential
AUTO  = os.environ.get("GATE_AUTO", "deny")   # safe default; explicitly set approve only for local simulation
def approves(a, args):
    print(f"   >> GATE: {a}({args}) — approve? [{AUTO}]"); return AUTO.lower().startswith("a")

SYSTEM = ("You are a clinical follow-up assistant working over FHIR. Output ONLY JSON "
          '{"thought","action","action_input"}. Actions: '+str(list(TOOLS)+["final"])+
          ". Read the INR and meds; if the INR is out of range, write_flag a dose-review request. Change no order.")
def parse(txt):
    """Parse the first JSON object and tolerate surrounding model commentary."""
    t = re.sub(r"^```[a-z]*|```$", "", txt.strip(), flags=re.M).strip()
    try:
        value = json.loads(t)
    except json.JSONDecodeError:
        start = t.find("{")
        if start < 0:
            raise ValueError(f"Model did not return a JSON object: {t[:200]!r}")
        value, _end = json.JSONDecoder().raw_decode(t[start:])
    if not isinstance(value, dict):
        raise ValueError("Model action must be a JSON object.")
    allowed_fields = {"thought", "rationale", "action", "action_input"}
    unexpected = set(value) - allowed_fields
    if unexpected:
        raise ValueError(f"Unexpected action fields: {sorted(unexpected)}")
    action = value.get("action")
    if not isinstance(action, str) or not action:
        raise ValueError("Model action must contain a non-empty string 'action'.")
    action_input = value.get("action_input", {})
    if not isinstance(action_input, dict):
        raise ValueError("Model action_input must be a JSON object.")
    value["action_input"] = action_input
    available_tools = set(globals().get("TOOLS", {}))
    if available_tools and action not in available_tools | {"final"}:
        allowed = sorted(available_tools | {"final"})
        raise ValueError(f"Action {action!r} is not allowed; expected one of {allowed}.")
    if action in {"change_order", "write_flag"}:
        has_text = any(isinstance(item, str) and item.strip() for item in action_input.values())
        if not has_text:
            raise ValueError(f"Action {action!r} requires a non-empty textual argument.")
    if action == "draft_followup" and not action_input:
        raise ValueError("Action 'draft_followup' requires a non-empty structured payload.")
    return value

msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":"Handle the anticoagulant follow-up."}]
for step in range(6):
    act=parse(chat(msgs)); print(f"[{step}] {act.get('thought','')}")
    a=act.get("action","final")
    if a=="final": print("    DONE:", act.get("action_input",{}).get("summary","")); break
    if a not in TOOLS: print(f"    (unknown action {a!r}; stopping)"); break
    args=act.get("action_input",{})
    obs = TOOLS[a](**args) if not (a in GATED and not approves(a,args)) else {"blocked":"human denied"}
    print("    OBS:", obs)
    msgs += [{"role":"assistant","content":json.dumps(act)},
             {"role":"user","content":f"[{a}] observation: {json.dumps(obs)}"}]


### Read back what we wrote


In [ ]:
print("Communications on the record:", BUNDLE["Communication"])


### Takeaway
The agent logic — perceive, plan, act, gated write — did not change; only the *environment* did, from a Python dict to FHIR resources. That is the whole point of building on standards: the same control-loop pattern can be adapted from a toy record to Synthea and later to a governed EHR sandbox. The production adapter, authorization, patient scoping, validation, and governance will change substantially. What Synthea gives you is the patient **state**; what this notebook adds is the **read/write interface**; the missing piece the paper flags is the **dynamics** (an action changing future state), which you build on top.

*To go live:* `git clone` Synthea, generate a population (`./run_synthea -p 100`), load the FHIR bundles into a HAPI FHIR server, then `export FHIR_BASE_URL=http://localhost:8080/fhir` and re-run the teaching adapter. Do not interpret this simplified connection as production-ready EHR integration.


## Exercise

No programming is required for this section.

**Activity:** Inspect the readable FHIR-derived summary and decide whether the agent has the correct patient, current observation, medication status, and sufficient context for a write-back. Specify what the flag should contain.

### Discussion prompts

- How do we know every resource belongs to the intended patient?
- What should happen if the newest observation lacks a value or date?
- Which production requirements are absent from this teaching adapter?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: decide whether a simulated write-back has enough context.
WRITEBACK = {
    "patient_confirmed": True,
    "source_observation_linked": True,
    "author_identified": False,
    "status": "preparation",
}
missing = [name for name, present in WRITEBACK.items() if present is False]
print("Proposed write-back:", WRITEBACK)
print("Missing controls:", missing or "none")
print("Workshop decision:", "HOLD" if missing else "eligible for the simulated gated path")


## Optional technical extension

Add patient-scoped searches, a Patient reference on the Communication, provenance, error handling, authentication placeholders, pagination, and an explicit read-only mode.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

FHIR standardizes data exchange, not the complete deployment. Production integration additionally requires patient scoping, profiles, terminology, authentication, authorization, consent, provenance, audit, workflow ownership, and validation.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
